# 02. SQL: загрузка в базу и аналитические срезы

1. Загружаем `data/raw/listings_*.csv` в SQLite `data/krisha.db`: таблицы `listings` и `districts` (район → берег).
2. Создаём представление `v_flats`: отсекаем неправдоподобные объявления, считаем цену за м², тип этажа.
3. Выполняем запросы из `sql/queries.sql`: JOIN, CTE, оконные функции (`ROW_NUMBER`, `RANK`, `PERCENT_RANK`, `SUM() OVER`).
4. Сохраняем витрины для этапов 3–7.

Рядом с ноутбуком должны лежать `build_db.py` и папка `sql/`.

In [1]:
import glob
import re
import sqlite3

import pandas as pd

from build_db import build

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

csv_path = sorted(glob.glob("data/raw/listings_*.csv"))[-1]
listings = build(csv_path, "data/krisha.db")
print(f"Загружено из {csv_path}: {len(listings)} объявлений")
print(f"Район не определён: {listings['district'].isna().sum()} ({listings['district'].isna().mean():.1%})")

sql_text = open("sql/queries.sql", encoding="utf-8").read()
blocks = re.split(r"^-- name:\s*", sql_text, flags=re.M)[1:]
QUERIES = {b.split("\n", 1)[0].strip(): b.split("\n", 1)[1] for b in blocks}

con = sqlite3.connect("data/krisha.db")
con.executescript(QUERIES["create_view_flats"])

def run(name):
    return pd.read_sql(QUERIES[name], con)

pd.read_sql("""
    SELECT l.deal, COUNT(*) AS было, (SELECT COUNT(*) FROM v_flats v WHERE v.deal = l.deal) AS после_фильтров
    FROM listings l GROUP BY l.deal
""", con)

Загружено из data/raw\listings_20261103.csv: 5755 объявлений
Район не определён: 134 (2.3%)


,deal,было,после_фильтров
0,rent,2814,2809
1,sale,2941,2940


## Объём предложения и цена за м² по районам

In [2]:
run("overview_by_district")

,deal,district,bank,listings,share_pct,median_price_per_m2
0,rent,Есильский,Левый берег,1134,40.4,5937.0
1,rent,Нура,Левый берег,717,25.5,6098.0
2,rent,Алматинский,Правый берег,399,14.2,5500.0
3,rent,Сарыарка,Правый берег,203,7.2,5102.0
4,rent,Сарайшык,Правый берег,199,7.1,5764.0
5,rent,Не указан,Не указан,90,3.2,5244.0
6,rent,Байконур,Правый берег,67,2.4,4815.0
7,sale,Нура,Левый берег,908,30.9,665726.0
8,sale,Есильский,Левый берег,893,30.4,753000.0
9,sale,Алматинский,Правый берег,376,12.8,543712.0


## Медианы по сегментам «район × комнаты»

In [3]:
run("segment_medians")

,deal,district,bank,rooms_grp,n,median_price_per_m2
0,rent,Алматинский,Правый берег,1,158,5750.0
1,sale,Алматинский,Правый берег,1,82,565417.0
2,rent,Алматинский,Правый берег,2,126,5000.0
3,sale,Алматинский,Правый берег,2,134,509167.0
4,rent,Алматинский,Правый берег,3,75,4605.0
5,sale,Алматинский,Правый берег,3,129,550351.0
6,rent,Алматинский,Правый берег,4,40,6505.0
7,sale,Алматинский,Правый берег,4,31,643440.0
8,rent,Байконур,Правый берег,1,30,5187.0
9,sale,Байконур,Правый берег,1,18,543664.0


## Доходность и окупаемость по сегментам

Валовая доходность = аренда за м² × 12 / цена за м². Показаны сегменты, где не меньше 10 объявлений продажи и аренды.

In [4]:
run("yield_by_segment")

,district,bank,rooms_grp,n_sale,n_rent,sale_price_per_m2,rent_per_m2_month,gross_yield_pct,payback_years,yield_rank
0,Сарыарка,Правый берег,1,96,90,510000.0,5520.0,12.99,7.7,1
1,Алматинский,Правый берег,1,82,158,565417.0,5750.0,12.20,8.2,2
2,Сарайшык,Правый берег,1,85,73,579196.0,5882.0,12.19,8.2,3
3,Алматинский,Правый берег,4,31,40,643440.0,6505.0,12.13,8.2,4
4,Алматинский,Правый берег,2,134,126,509167.0,5000.0,11.78,8.5,5
5,Нура,Левый берег,1,192,241,644387.0,6250.0,11.64,8.6,6
6,Байконур,Правый берег,2,26,21,473154.0,4545.0,11.53,8.7,7
7,Байконур,Правый берег,1,18,30,543664.0,5187.0,11.45,8.7,8
8,Сарыарка,Правый берег,2,91,74,500000.0,4734.0,11.36,8.8,9
9,Сарайшык,Правый берег,2,141,74,597470.0,5544.0,11.13,9.0,10


## H1. Доходность по числу комнат

In [5]:
run("h1_rooms_yield")

,rooms_grp,n,q1_yield,median_yield,q3_yield
0,1,628,10.05,11.53,13.26
1,2,1069,8.92,10.60,12.40
2,3,919,7.86,9.53,11.38
3,4,263,7.36,9.46,12.67


## H2. Левый и правый берег

In [6]:
run("h2_bank_comparison")

,bank,n_sale,n_rent,sale_price_per_m2,rent_per_m2_month,gross_yield_pct,payback_years
0,Правый берег,1095,868,540000.0,5370.0,11.93,8.4
1,Левый берег,1801,1851,707547.0,6000.0,10.18,9.8


## H3. Дисконт крайних этажей

Отклонение цены за м² от средних этажей в том же районе и с тем же числом комнат.

In [7]:
run("h3_floor_discount")

,floor_type,segments,listings,weighted_diff_vs_middle_pct
0,Первый,10,69,-9.8
1,Последний,18,274,-10.5


## Что можно купить с бюджетом 20–50 млн ₸

In [8]:
run("budget_segments")

,budget,district,listings,share_in_budget_pct
0,1) до 20 млн,Сарыарка,58,34.5
1,1) до 20 млн,Нура,35,20.8
2,1) до 20 млн,Алматинский,27,16.1
3,1) до 20 млн,Сарайшык,23,13.7
4,1) до 20 млн,Есильский,16,9.5
5,1) до 20 млн,Байконур,7,4.2
6,1) до 20 млн,Не указан,2,1.2
7,2) 20–35 млн,Нура,307,31.1
8,2) 20–35 млн,Алматинский,181,18.3
9,2) 20–35 млн,Есильский,165,16.7


## Предварительно: самые дешёвые 10% в своём сегменте

In [9]:
run("cheap_vs_segment")

,listing_id,district,rooms,area_m2,floor,floors_total,price_kzt,price_per_m2,pct_rank,seg_size
0,1016946524,Алматинский,1,49.0,1.0,4.0,17500000,357143.0,0.000000,82
1,1016812739,Алматинский,2,91.7,3.0,5.0,24000000,261723.0,0.000000,134
2,1004035673,Алматинский,3,99.1,11.0,13.0,34000000,343088.0,0.000000,129
3,1002935073,Алматинский,4,175.0,9.0,13.0,50500000,288571.0,0.000000,31
4,1012530169,Алматинский,2,76.0,1.0,6.0,23000000,302632.0,0.007519,134
5,1016299628,Алматинский,3,101.8,3.0,18.0,35900000,352652.0,0.007812,129
6,1014535258,Алматинский,1,46.0,1.0,6.0,18000000,391304.0,0.012346,82
7,1014587023,Алматинский,2,45.0,1.0,5.0,16000000,355556.0,0.015038,134
8,1013645193,Алматинский,3,83.3,6.0,6.0,29500000,354142.0,0.015625,129
9,694550956,Алматинский,2,79.0,3.0,9.0,29000000,367089.0,0.022556,134


## Витрины для следующих этапов

In [10]:
flats = pd.read_sql("SELECT * FROM v_flats", con)
flats.to_csv("data/flats.csv", index=False, encoding="utf-8-sig")

listing_yield = run("listing_yield")
listing_yield.to_csv("data/listing_yield.csv", index=False, encoding="utf-8-sig")

print(f"flats.csv: {len(flats)} строк (продажа и аренда)")
print(f"listing_yield.csv: {len(listing_yield)} квартир на продажу с оценкой доходности")

flats.csv: 5749 строк (продажа и аренда)
listing_yield.csv: 2879 квартир на продажу с оценкой доходности
